# Chapter 8 · Meet a real one (optional)

Your tinyGPT has 10.7 million parameters and has read 1 MB of Shakespeare. **GPT-2
small**, released by OpenAI in 2019, has 124 million parameters and read about 40 GB of
web pages. In this chapter you put the two side by side.

Comparing two language models fairly is harder than it looks: they don't even read
text in the same pieces.

| | |
|---|---|
| **Open box** | how a model reads a text: `token_surprises` in `tinygpt/compare.py` |
| **The number** | tinyGPT against GPT-2, per character |
| **Time** | about 45 minutes |
| **Python you need** | lists and functions |
| **GPU** | recommended; GPT-2 also runs on a CPU, more slowly |
| **Download** | GPT-2 is about 500 MB, downloaded from Hugging Face the first time |

**How this chapter works.** Five steps, in order:

1. **Predict** `everyone`: guess before you run anything. A wrong guess is useful: it shows you what you expected.
2. **Run** `everyone`: run the cells and compare with your guesses.
3. **Investigate** `everyone`: open the box. Read the code that just ran and answer a few questions.
4. **Modify** `most people`: change one thing, predict what will happen, then explain what did.
5. **Make** `optional challenge`: build this chapter's piece yourself. A check tells you when it's right.

Run a cell with **Shift+Enter**. If the notebook restarts, run the setup cell again first.

## Setup

In [ ]:
#@title Setup: run this cell first (about 30 seconds) { display-mode: "form" }
# Downloads tinyGPT (on Colab) and installs it. You don't need to read or change this cell.
import importlib, os, subprocess, sys
from pathlib import Path

def run(*command):
    done = subprocess.run(command, capture_output=True, text=True)
    if done.returncode != 0:
        raise RuntimeError(f"{' '.join(command)} failed:\n{done.stdout}{done.stderr}")

if "google.colab" in sys.modules:
    root = Path("/content/tinyGPT")
    if not root.exists():
        run("git", "clone", "-q", "https://github.com/cgpan/tinyGPT.git", str(root))
else:
    here = Path.cwd().resolve()
    root = next((p for p in [here, *here.parents] if (p / "src" / "tinygpt").is_dir()), None)
    if root is None:
        raise FileNotFoundError("Can't find the tinyGPT folder. Open this notebook from inside it, or use Colab.")
os.chdir(root)
run(sys.executable, "-m", "pip", "install", "-q", ".")
importlib.invalidate_caches()

import tinygpt
tinygpt.print_setup()

### The two models

Your tinyGPT from chapter 5, and GPT-2 small from Hugging Face, a website that hosts
trained models. The download takes a minute or so the first time.

In [ ]:
import torch

model, tok, history = tinygpt.load_kept_model()
gpt2, gpt2_tok = tinygpt.compare.load_gpt2()
text = tinygpt.load_text()
val_text = text[int(len(text) * 0.9):]           # the validation text, as in every chapter

print(f"tinyGPT: {tinygpt.count_params(model):>11,} parameters, {tok.vocab_size:>6,} tokens in its vocabulary")
print(f"GPT-2:   {tinygpt.count_params(gpt2):>11,} parameters, {len(gpt2_tok):>6,} tokens in its vocabulary")

## 1 · Predict `everyone`

GPT-2's tokens are not characters. Its vocabulary of 50,257 tokens is mostly pieces of
words, chosen because they are common in web text.

In [ ]:
guess_gpt2_tokens = None     # "ROMEO:\nBut, soft! what light through yonder window breaks?" is 58
                             # characters. How many GPT-2 tokens is it?
guess_gpt2_per_token = None  # GPT-2's average surprise per *token* on our validation text:
                             # above or below tinyGPT's 1.49?
guess_winner = None          # per *character*, which model is less surprised by our Shakespeare?

## 2 · Run `everyone`

How each model splits the same line:

In [ ]:
line = "ROMEO:\nBut, soft! what light through yonder window breaks?"
gpt2_ids = gpt2_tok.encode(line)
print(f"tinyGPT: {len(tok.encode(line))} tokens, one per character")
print(f"GPT-2:   {len(gpt2_ids)} tokens: {[gpt2_tok.decode([i]) for i in gpt2_ids]}")

Now the number: each model's average surprise on the whole validation text, counted
two ways. This takes a few seconds on a GPU, or a minute or two on a CPU.

In [ ]:
tiny_per_token, tiny_per_char = tinygpt.compare.per_token_and_per_character(val_text, model, tok)
gpt2_per_token, gpt2_per_char = tinygpt.compare.per_token_and_per_character(val_text, gpt2, gpt2_tok)

print(f"{'':10}{'per token':>12}{'per character':>16}")
print(f"{'tinyGPT':10}{tiny_per_token:>12.3f}{tiny_per_char:>16.3f}")
print(f"{'GPT-2':10}{gpt2_per_token:>12.3f}{gpt2_per_char:>16.3f}")
print("your guesses:", guess_gpt2_tokens, guess_gpt2_per_token, guess_winner)

## 3 · Investigate `everyone`

### Open the box

In [ ]:
tinygpt.show(tinygpt.compare.token_surprises)

1. `tokenizer.encode(text)`: each model splits the text its own way. Why is `window`
   128 for tinyGPT but 1,024 for GPT-2? About how many *characters* back can each one read?
2. The `tokens` and `surprises` lists line up one to one. In the next cell, where is
   each model most surprised in our line? Least?
3. Both models' totals are divided two ways in `per_token_and_per_character`: by the
   number of tokens, and by the number of characters. Which one lets you compare models
   that use different tokens?

In [ ]:
for name, (m, t) in {"tinyGPT": (model, tok), "GPT-2": (gpt2, gpt2_tok)}.items():
    tokens, surprises = tinygpt.compare.token_surprises(line, m, t)
    print(name, " ".join(f"{token!r}:{s:.1f}" for token, s in zip(tokens, surprises)), "\n")

### The units trap

A GPT-2 token covers several characters, so its surprise at one token is the surprise of
several characters at once. Per token, GPT-2's number *has* to be bigger. Comparing
GPT-2's per-token number with tinyGPT's per-character number is like comparing a price
per dozen with a price per egg.

In [ ]:
chars_per_token = len(val_text) / len(gpt2_tok.encode(val_text))
print(f"one GPT-2 token covers {chars_per_token:.2f} characters of our Shakespeare, on average")
print(f"GPT-2 per token / characters per token = {gpt2_per_token / chars_per_token:.3f}  "
      f"(per character: {gpt2_per_char:.3f})")

Per character is fair on units. Is the comparison fair in other ways? Think about how far
back each model can read; whether GPT-2 might have read Shakespeare somewhere on the web;
that tinyGPT trained on *this* file's layout, with names in capitals and blank lines; and
that GPT-2 is twelve times bigger.

*Your explanation (double-click to edit):*

## 4 · Modify `most people`

**A. Modern English.** Write a few sentences of your own, or use the example. Use only
tinyGPT's 65 characters: no digits except 3, no accents, and straight quotes (`'`, not
`’`). **Predict** which model will be less surprised, per character.

In [ ]:
modern = ("Dear students, the deadline for the project has moved to Friday. Please upload your "
          "notebook and a short report, and come to office hours if you have any questions about it.")
for name, (m, t) in {"tinyGPT": (model, tok), "GPT-2": (gpt2, gpt2_tok)}.items():
    print(f"{name:8} per character: {tinygpt.compare.per_token_and_per_character(modern, m, t)[1]:.3f}")

How big is the gap here, compared with the gap on Shakespeare? Why?

*Your explanation (double-click to edit):*

**B. Let them both write.** Same start, same temperature. Which one writes more like
the plays in our file? Which one writes more like ordinary English?

> **A warning before you run it.** GPT-2 learned from unfiltered web pages, and at
> temperature 1.0 it sometimes writes offensive or sexually explicit text; one of our test
> runs did. That is one reason today's chat models get extra safety training before
> release. Skip this step if you would rather not see unfiltered output.

In [ ]:
print("----- tinyGPT -----")
print(tinygpt.sample(model, tok, prompt="JULIET:\n", length=300, seed=1))
print("----- GPT-2 -----")
print(tinygpt.compare.gpt2_sample(gpt2, gpt2_tok, prompt="JULIET:\n", length=80, seed=1))

*Your explanation (double-click to edit):*

## 5 · Make `optional challenge`

Write the fair average yourself: given the tokens a model predicted and its surprise at
each one, return the average surprise **per character**.

In [ ]:
def my_per_character(tokens, surprises):
    """tokens: a list of strings. surprises: a list of numbers, one per token."""
    return 0.0                    # replace this line

tinygpt.check.per_character(my_per_character)

When the check passes, use it on GPT-2's real surprises. It should match the table in step 2.

In [ ]:
tokens, surprises = tinygpt.compare.token_surprises(val_text, gpt2, gpt2_tok)
print(f"GPT-2 per character, your way: {my_per_character(tokens, surprises):.3f}")

Going further: find a passage of at least 200 characters, using only tinyGPT's 65
characters, on which tinyGPT beats GPT-2 per character, or one where it loses by the
most. What do those passages have in common?

**Stuck?** Try first, then compare with [the solution](https://github.com/cgpan/tinyGPT/blob/main/solutions/08_meet_gpt2.md) (also in the `solutions/` folder of your copy).

## Where the number stands

In [ ]:
tinygpt.scoreboard({"tinyGPT, per character": tiny_per_char,
                    "GPT-2, per character": gpt2_per_char,
                    "GPT-2, per token (not comparable)": gpt2_per_token})

## Common questions

**How does GPT-2 choose its tokens?** With *byte-pair encoding*: start from single
characters and repeatedly merge the pair that appears most often in the training text,
until there are 50,257 tokens. Common words become one token; rare words are spelled out
in pieces.

**Why not train GPT-2 ourselves?** It was trained on about 40 GB of text, some 36,000
times our file, with far more computing power than a notebook has. tinyGPT shows every
idea at a size that trains in minutes.

**Is GPT-2 like ChatGPT?** It is an ancestor: the same decoder-only transformer design
you built. Chat models are far bigger, trained on far more text, and then trained further
to follow instructions and hold a conversation.

**Did GPT-2 read Shakespeare?** Probably some: Shakespeare is all over the web. Nobody
can easily check what is in 40 GB of training text. When a model is tested on text it may
have seen, that is called *contamination*, and it is a real problem in judging today's
models.

## If something breaks

| What you see | What to do |
|---|---|
| The download fails, or can't reach huggingface.co | Check your internet connection and run the cell again. Some school or office networks block Hugging Face; Colab doesn't. |
| `ValueError: ... is not in the vocabulary` | Your text has a character tinyGPT can't read. Text pasted from a document often has curly quotes (’ “ ”); replace them with straight ones. |
| GPT-2 is slow | It is 12 times bigger than tinyGPT. On a CPU, measuring the validation text takes a minute or two. |

## What next?

You have built, trained, tested and measured a GPT. Some directions from here:

- **Train on a different text:** another author, song lyrics, or code. Change
  `load_text`, and set `vocab_size` to the number of distinct characters in the new text.
- **Use GPT-2's tokens** instead of characters for tinyGPT, and compare per character.
- **Add a learning-rate schedule:** start fast and slow down. Does the number improve?
- **Batch the heads:** compute all heads of a block with one matrix multiplication. Same
  maths, usually faster, and good practice in reading tensor shapes.